# 🧠 CogniFusion — Colab Notebook

**Real-time cognitive state monitoring** fusing EEG + eye-tracking to classify
**low / medium / high** cognitive load.

| | |
|---|---|
| 📓 Repo | [PoornaSri26/CogniFusion](https://github.com/PoornaSri26/CogniFusion) |
| ⚡ Runtime | CPU is fine for the demo · GPU speeds up training |
| 🗄️ Data | Synthetic 30-subject dataset generated in-notebook (no hardware needed) |

**Pipeline:** synthetic data → fused EEG+eye features → AttentionFusionNet
(vector-gated cross-modal fusion, focal loss, EMA smoothing) → evaluation → live inference demo.

## 1 · Setup — clone repo & install deps

In [ ]:
# Clone the repository
!git clone -q https://github.com/PoornaSri26/CogniFusion.git
%cd CogniFusion

# Core ML deps. torch is preinstalled on Colab — don't reinstall it.
!pip install -q numpy pandas scipy scikit-learn pyarrow

import sys, torch
sys.path.insert(0, "src")
sys.path.insert(0, "data")
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 2 · Generate the synthetic dataset

30 subjects × 4 epochs × 3 classes = **360 synchronized EEG+eye epochs**.
Each subject gets unique neural parameters (IAF 9–12 Hz, noise floor,
electrode offsets, pupil baseline) so the model learns subject-invariant features.

In [ ]:
from simulate_data import generate_dataset
manifest = generate_dataset("data/raw")
manifest.head()

## 3 · Build the fused feature table

EEG features (band powers, differential entropy, connectivity, entropy) + eye features (pupillometry, fixations, blinks, gaze kinematics) → one parquet table.

In [ ]:
from build_dataset import build
df = build("data/raw/manifest.csv", "data/processed/features.parquet")
df.shape

## 4 · Quick EDA

In [ ]:
import pandas as pd

df = pd.read_parquet("data/processed/features.parquet")
print("Label distribution:")
display(df["label"].value_counts().to_frame("count"))

feat_cols = [c for c in df.columns if c.startswith(("eeg__", "eye__"))]
print(f"
{len(feat_cols)} fused features | missing values: {int(df[feat_cols].isna().sum().sum())}")

# Which features correlate most with load (0=low, 1=medium, 2=high)?
load = df["label"].map({"low": 0, "medium": 1, "high": 2})
top = df[feat_cols].corrwith(load).abs().sort_values(ascending=False).head(10)
print("
Top features by |correlation| with cognitive load:")
display(top.to_frame("|r|"))

## 5 · Train the model

Single subject-wise holdout by default (fast). For a rigorous estimate, add `--cv 5`
(5-fold subject-wise cross-validation — trains 5 models, ~5× slower).

In [ ]:
# ~200 epochs on 360 samples — a few minutes on CPU, faster on GPU.
# Bump to the full 300 or add --cv 5 for the final run.
!python src/train.py --features data/processed/features.parquet                     --epochs 200 --cv 0                     --out models/fusion_model

In [ ]:
import torch, json
ckpt = torch.load("models/fusion_model.pt", map_location="cpu", weights_only=False)
print(json.dumps(ckpt["cv_summary"], indent=2))

## 6 · Live inference demo

Loads the self-contained ensemble checkpoint and predicts cognitive load on
held-out epochs — same path the real-time pipeline uses (per-fold scalers,
vector-gated attention fusion, probability averaging).

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

from eeg_features import extract_eeg_features
from eye_features import extract_eye_features
from model import AttentionFusionNet

CHANNEL_NAMES = ["AF7", "AF8", "TP9", "TP10"]

def load_ensemble(path="models/fusion_model.pt"):
    """Load the self-contained ensemble checkpoint (same logic as realtime_pipeline.load_model,
    without the heavy hardware deps like brainflow)."""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    folds = []
    for rec in ckpt["folds"]:
        m = AttentionFusionNet(
            eeg_dim=len(ckpt["eeg_cols"]),
            eye_dim=len(ckpt["eye_cols"]),
            embed_dim=rec.get("embed_dim", ckpt.get("embed_dim", 48)),
            n_classes=len(ckpt["label_classes"]),
            gate_type=ckpt.get("gate_type", "vector"),
        )
        m.load_state_dict(rec["model_state"])
        m.eval()
        folds.append({"model": m, "scaler_eeg": rec["scaler_eeg"], "scaler_eye": rec["scaler_eye"]})
    return {"folds": folds, "eeg_cols": ckpt["eeg_cols"],
            "eye_cols": ckpt["eye_cols"], "label_classes": ckpt["label_classes"]}

bundle = load_ensemble("models/fusion_model.pt")
eeg_cols, eye_cols, classes = bundle["eeg_cols"], bundle["eye_cols"], bundle["label_classes"]

def predict_epoch(row):
    eeg = np.load(row["eeg_path"])                    # (4 channels, 1024 samples)
    eye = pd.read_csv(row["eye_path"])
    ef = {f"eeg__{k}": float(v) for k, v in
          extract_eeg_features(eeg, fs=256, channel_names=CHANNEL_NAMES).items()}
    xf = {f"eye__{k}": float(v) for k, v in extract_eye_features(eye).items()}
    eeg_vec = pd.Series({c: ef.get(c, 0.0) for c in eeg_cols}, dtype=float)
    eye_vec = pd.Series({c: xf.get(c, 0.0) for c in eye_cols}, dtype=float)
    eeg_in = eeg_vec.reindex(eeg_cols).fillna(0.0).to_numpy()[None, :]
    eye_in = eye_vec.reindex(eye_cols).fillna(0.0).to_numpy()[None, :]

    probs = np.zeros(len(classes))
    with torch.no_grad():
        for fold in bundle["folds"]:
            xe = fold["scaler_eeg"].transform(eeg_in)
            ye = fold["scaler_eye"].transform(eye_in)
            logits, _ = fold["model"](torch.tensor(xe, dtype=torch.float32),
                                      torch.tensor(ye, dtype=torch.float32))
            probs += torch.softmax(logits, dim=1).numpy()[0]
    return probs / len(bundle["folds"])

# Balanced sample across all classes
manifest = pd.read_csv("data/raw/manifest.csv")
sample = (manifest.groupby("label", group_keys=False)
          .apply(lambda g: g.sample(min(10, len(g)), random_state=0))
          .reset_index(drop=True))

preds = [predict_epoch(r) for _, r in sample.iterrows()]
sample["pred"]       = [classes[int(p.argmax())] for p in preds]
sample["confidence"] = [round(float(p.max()), 3) for p in preds]

display(sample[["subject", "label", "pred", "confidence"]].head(15))

acc = accuracy_score(sample["label"], sample["pred"])
print(f"
Accuracy on {len(sample)} sampled epochs: {acc:.1%}")
ConfusionMatrixDisplay(
    confusion_matrix(sample["label"], sample["pred"], labels=classes),
    display_labels=classes).plot()
plt.title("CogniFusion — confusion matrix (sampled epochs)")
plt.show()

## 7 · Next steps

- **Real data**: `python src/build_dataset_real.py` builds the fused table from the
  1.5 GB real Muse-EEG + webcam dataset (`data/raw/dataset.csv`, 113 participants —
  see the repo README for how to obtain it), then train with
  `--features data/processed/features_real.parquet`.
- **Live demo**: run `python src/realtime_pipeline.py --eeg synthetic --eye synthetic`
  locally for a synthetic real-time session, or `streamlit run dashboard.py` for the
  monitoring dashboard. Webcam eye-tracking works in Colab via
  `src/webcam_eyetracker.py`, but EEG needs Muse hardware (BrainFlow).
- **Full training**: `--epochs 300 --cv 5` for the production-grade subject-wise CV estimate.